# Guided Solver Config Builder

Configure your model inputs and write `solver_params.json` to disk.
Run this notebook before `guided_bayesian_inference.ipynb`.

In [ ]:
import sys
sys.path.append("../")
from solver_config_builder import build_solver_params_config, write_solver_params_json_file

In [ ]:
# =======================
# FILE PATHS
# =======================
# name = "ACP"
# name = "MalCoA"
name = "Both_Normal"

# Use a reaction folder containing one or more reaction JSON files.
reactions_source = "./Reactions"

solver_file_dir = "."
solver_params_file = f"solver_params_{name}.json"

# Set save directory and output file names
savedir = "./Results"
prior_samples_file = f"{name}_prior_samples_pm.nc"
posterior_samples_file = f"{name}_Model_posterior_samples_pm.nc"
trace_plot_file = f"{name}_Model_trace_plot.png"

# =============================
# FREE PARAMETERS
# =============================
# Required fields: rxn_name, param_name, distribution, lower, upper
# Optional fields: mass (default: 0.95), fixed_stat
free_parameter_prior_inputs = [
    {"rxn_name": "FabD_binding_MalCoA", "param_name": "k2_1f", "distribution": "LogNormal", "lower": 0.0081, "upper": 8.8, "mass": 0.95},
    {"rxn_name": "FabD_binding_ACP", "param_name": "k2_3f", "distribution": "LogNormal", "lower": 0.00088, "upper": 0.28, "mass": 0.95},
    # {"rxn_name": "FabD_activation", "param_name": "k2_2f", "distribution": "Normal", "lower": 1e-3, "upper": 1e1, "mass": 0.95},
    # {"rxn_name": "FabD_catalysis", "param_name": "k2_4f", "distribution": "Normal", "lower": 1e-3, "upper": 1e1, "mass": 0.95},

]

# ========================
# DATASETS
# ========================
# This fixture set is designed to exercise all supported dataset types and noise models.
dataset_inputs = [
    {
        "name": "conc_vs_final_MalACP",
        "dataset_type": "endpoint",
        "data_file": "./Data/conc_vs_final_MalACP.csv",
        "observable": "MalACP (uM)",
        "time_values": [150],
        "column_mapping": {"MalACP (uM)": "MalACP (uM)"},
        "init_cond_columns": {"FabD": "FabD (uM)", "C3_MalCoA": "MalCoA (uM)", "ACP": "ACP (uM)"},
        "noise_model": "relative_mean",
        "noise_params": {"frac": 0.05},
    },
    {
        "name": "time_vs_MalACP",
        "dataset_type": "timeseries",
        "data_file": "./Data/time_vs_MalACP.csv",
        "observable": "MalACP (uM)",
        "time_column": "Time (ms)",
        "column_mapping": {"MalACP (uM)": "MalACP (uM)"},
        "noise_model": "relative_mean",
        "noise_params": {"frac": 0.05},
    },
]

# =====================================
# SAMPLING + SOLVER SETTINGS
# =====================================
prior_sampling_settings = {"draws": 1000, "random_seed": 0}
posterior_sampling_settings = {
    "draws": 200,
    "tune": 300,
    "chains": 4,
    "random_seed": 0,
    "target_accept": 0.95,
    "nuts_sampler": "nutpie"
}

ode_solver_settings = {
    "solver_name": "Kvaerno5",
    "max_steps": 10000000,
    "dt0": 1e-12,
    "stepsize_controller": "PIDController",
}

ode_stepsize_controller_settings = {
    "rtol": 1.0e-6,
    "atol": 1.0e-8,
    "pcoeff": 0.3,
    "icoeff": 0.4,
    "dcoeff": 0.0,
}

# =================================
# MODEL-SPECIFIC SETTINGS
# =================================
calculation_module_path = "./FabD_calculations.py"
initial_conditions = {"FabD": 1, "C3_MalCoA": 1000, "ACP": 10}

In [ ]:
solver_params = build_solver_params_config(
    free_parameter_prior_inputs=free_parameter_prior_inputs,
    dataset_inputs=dataset_inputs,
    prior_sampling_settings=prior_sampling_settings,
    posterior_sampling_settings=posterior_sampling_settings,
    ode_solver_settings=ode_solver_settings,
    ode_stepsize_controller_settings=ode_stepsize_controller_settings,
    calculation_module_path=calculation_module_path,
    initial_conditions=initial_conditions,
)

written_solver_params_path = write_solver_params_json_file(
    solver_params_config=solver_params,
    file_directory=solver_file_dir,
    filename=solver_params_file,
)

print(f"Wrote solver config: {written_solver_params_path}")
print(f"Reaction source: {reactions_source}")
print(f"Save directory: {savedir}")
print(
    f"Configured free params: {[param['param_name'] for param in solver_params['free_kinetic_params']]}"
)
print(
    f"Configured datasets: {[dataset['name'] for dataset in solver_params['datasets']]}"
)

Wrote solver config: /Users/annettethompson/Library/CloudStorage/OneDrive-SharedLibraries-UCB-O365/Jerome Michael Fox - Annie Thompson/Git Repositories/Bayesian Kinetic Model/Restructured Framework/Simplified_FAS/solver_params_Both_Normal.json
Reaction source: ./Reactions
Save directory: ./Results
Configured free params: ['k2_1f', 'k2_3f']
Configured datasets: ['conc_vs_final_MalACP', 'time_vs_MalACP']
